# List new event files to ingest

Attach `lh_meridian_hr` as the default lakehouse. In Fabric, mark Cell 2 as the parameter cell.

This notebook drives incremental ingestion of the monthly `workforce_events_YYYY-MM.csv`
extracts:

1. Read the current watermark from `bronze.ingestion_watermark` (a very old default on the
   first run, so every file qualifies).
2. List the source files in the GitHub repository through the Contents API.
3. Keep only files whose month is later than the watermark, sorted and capped at
   `max_files_per_run`.
4. Exit a JSON payload the pipeline consumes: `files` (relative Copy paths for the ForEach),
   `watermark` (the month to store after the batch succeeds), and `count`.

In [ ]:
pipeline_name = "workforce_events"
github_owner = "modamin"
github_repo = "fabric-developer-training-lab"
github_branch = "main"
github_path = "data/events"
copy_base_prefix = "events/"
default_watermark = "2020-12-01 00:00:00"
max_files_per_run = 60

In [ ]:
import json
import re
from datetime import datetime

import requests
from pyspark.sql.functions import col

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (
    pipeline_name STRING,
    watermark_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_row = (
    spark.table("bronze.ingestion_watermark")
    .where(col("pipeline_name") == pipeline_name)
    .select("watermark_timestamp")
    .collect()
)
if watermark_row and watermark_row[0]["watermark_timestamp"] is not None:
    watermark = watermark_row[0]["watermark_timestamp"]
else:
    watermark = datetime.strptime(default_watermark, "%Y-%m-%d %H:%M:%S")
watermark_month = watermark.strftime("%Y-%m")

api_url = f"https://api.github.com/repos/{github_owner}/{github_repo}/contents/{github_path}"
response = requests.get(api_url, params={"ref": github_branch}, timeout=60)
response.raise_for_status()

name_pattern = re.compile(r"^workforce_events_(\d{4}-\d{2})\.csv$")
new_files = []
for entry in response.json():
    if entry.get("type") != "file":
        continue
    match = name_pattern.match(entry["name"])
    if not match:
        continue
    file_month = match.group(1)
    if file_month > watermark_month:
        new_files.append((file_month, copy_base_prefix + entry["name"]))

new_files.sort()
new_files = new_files[:max_files_per_run]

files = [path for _, path in new_files]
if new_files:
    new_watermark = new_files[-1][0] + "-01 00:00:00"
else:
    new_watermark = watermark.strftime("%Y-%m-%d %H:%M:%S")

result = {
    "files": files,
    "watermark": new_watermark,
    "count": len(files),
}
print(result)
notebookutils.notebook.exit(json.dumps(result))